# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inshrah-malik/inshrah-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
from datasets import load_dataset
import pandas as pd
from itertools import islice

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

rows = list(islice(ds, 10000))
df = pd.DataFrame(rows)

print(df.shape)
df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

(10000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


In [21]:
df.shape

(10000, 30)

In [22]:
df.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events'],
      dtype='object')

In [23]:
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0, 1)
)

In [24]:
df = df[df["gsc_impressions"] > 0]

In [25]:
df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=[0,5,10,20,50,100],
    labels=[
        "1-5",
        "6-10",
        "11-20",
        "21-50",
        "51-100"
    ]
)

In [26]:
table1 = (
    df.groupby("position_bucket")
      .agg(
          n=("ctr","count"),
          avg_ctr=("ctr","mean")
      )
)

table1

/tmp/ipykernel_6288/2767344581.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("position_bucket")


,n,avg_ctr
position_bucket,,
1-5,923,0.031080
6-10,2856,0.011626
11-20,1832,0.011343
21-50,2874,0.005468
51-100,1475,0.000948


In [27]:
df["impression_bucket"] = pd.qcut(
    df["gsc_impressions"],
    q=4,
    duplicates="drop"
)

In [28]:
table2 = (
    df.groupby("impression_bucket")
      .agg(
          n=("ctr","count"),
          avg_ctr=("ctr","mean")
      )
)

table2

/tmp/ipykernel_6288/2485005550.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("impression_bucket")


,n,avg_ctr
impression_bucket,,
"(0.999, 2.0]",2772,0.010642
"(2.0, 6.0]",2483,0.012196
"(6.0, 14.0]",2349,0.010305
"(14.0, 506.0]",2396,0.008679


In [29]:
df["baseline_score"] = (
    df["gsc_impressions"] *
    (1 - df["ctr"])
)

In [30]:
df["reason_code"] = "HIGH_IMPRESSIONS_LOW_CTR"

In [31]:
df["action"] = "Review Title and Meta Description"

In [32]:
ranked = df.sort_values(
    "baseline_score",
    ascending=False
)

In [33]:
import os

os.makedirs(
    "../outputs",
    exist_ok=True
)

In [34]:
ranked.to_csv(
    "../outputs/baseline_action_score.csv",
    index=False
)

In [35]:
import os
os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

In [36]:
top10 = ranked.head(10)

top10[
    [
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "baseline_score",
        "action"
    ]
]

,content_hash_id,gsc_impressions,gsc_clicks,ctr,baseline_score,action
6854,content_4e8d1e11f60fe6ba,506,11,0.021739,495.0,Review Title and Meta Description
9596,content_4e8d1e11f60fe6ba,466,2,0.004292,464.0,Review Title and Meta Description
3825,content_213eb91f21a43550,424,0,0.000000,424.0,Review Title and Meta Description
5546,content_213eb91f21a43550,324,0,0.000000,324.0,Review Title and Meta Description
1221,content_f94fe855380e150f,305,5,0.016393,300.0,Review Title and Meta Description
446,content_f94fe855380e150f,303,3,0.009901,300.0,Review Title and Meta Description
146,content_f94fe855380e150f,266,6,0.022556,260.0,Review Title and Meta Description
5607,content_f94fe855380e150f,249,5,0.020080,244.0,Review Title and Meta Description
1019,content_f94fe855380e150f,247,6,0.024291,241.0,Review Title and Meta Description
766,content_f94fe855380e150f,240,5,0.020833,235.0,Review Title and Meta Description


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule:

Prioritize pages that have high search impressions but low click-through rate (CTR). These pages already receive good visibility in Google Search but are not attracting enough clicks, making them strong candidates for title and meta description optimization.

Score:

Baseline Score = gsc_impressions × (1 − CTR)

Higher scores indicate higher priority for review.

Reason Code:

HIGH_IMPRESSIONS_LOW_CTR

Action:

Review Title and Meta Description

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [37]:
import os
import pandas as pd

# Create CTR
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"].replace(0, 1)

# Remove rows with zero impressions
df = df[df["gsc_impressions"] > 0].copy()

# Baseline score
df["baseline_score"] = df["gsc_impressions"] * (1 - df["ctr"])

# Reason code
df["reason_code"] = "HIGH_IMPRESSIONS_LOW_CTR"

# Action label
df["action"] = "Review Title and Meta Description"

# Rank pages
ranked = df.sort_values("baseline_score", ascending=False)

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Save CSV
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("CSV saved successfully!")

ranked.head(10)

CSV saved successfully!


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_claude,ai_meta,ai_other,scroll_events,ctr,position_bucket,impression_bucket,baseline_score,reason_code,action
6854,2025-02-12,client_73cda7b4e4f265ea,content_4e8d1e11f60fe6ba,True,True,True,False,506,11,3565.0,...,0,0,0,0,0.021739,6-10,"(14.0, 506.0]",495.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
9596,2025-02-13,client_73cda7b4e4f265ea,content_4e8d1e11f60fe6ba,True,True,True,False,466,2,3495.0,...,0,0,0,0,0.004292,6-10,"(14.0, 506.0]",464.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
3825,2025-02-11,client_9958f0a7ae1df715,content_213eb91f21a43550,True,True,True,False,424,0,466.0,...,0,0,0,0,0.000000,1-5,"(14.0, 506.0]",424.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
5546,2025-02-12,client_9958f0a7ae1df715,content_213eb91f21a43550,True,True,True,False,324,0,627.0,...,0,0,0,0,0.000000,1-5,"(14.0, 506.0]",324.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
1221,2025-01-31,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,305,5,484.0,...,0,0,0,0,0.016393,1-5,"(14.0, 506.0]",300.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
446,2025-01-28,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,303,3,549.0,...,0,0,0,0,0.009901,1-5,"(14.0, 506.0]",300.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
146,2025-01-27,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,266,6,482.0,...,0,0,0,0,0.022556,1-5,"(14.0, 506.0]",260.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
5607,2025-02-12,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,249,5,529.0,...,0,0,0,0,0.020080,1-5,"(14.0, 506.0]",244.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
1019,2025-01-30,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,247,6,409.0,...,0,0,0,0,0.024291,1-5,"(14.0, 506.0]",241.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
766,2025-01-29,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,240,5,434.0,...,0,0,0,0,0.020833,1-5,"(14.0, 506.0]",235.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [38]:
top20 = ranked.head(20)

top20[
    [
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

,content_hash_id,gsc_impressions,gsc_clicks,ctr,baseline_score,reason_code,action
6854,content_4e8d1e11f60fe6ba,506,11,0.021739,495.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
9596,content_4e8d1e11f60fe6ba,466,2,0.004292,464.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
3825,content_213eb91f21a43550,424,0,0.000000,424.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
5546,content_213eb91f21a43550,324,0,0.000000,324.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
1221,content_f94fe855380e150f,305,5,0.016393,300.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
446,content_f94fe855380e150f,303,3,0.009901,300.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
146,content_f94fe855380e150f,266,6,0.022556,260.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
5607,content_f94fe855380e150f,249,5,0.020080,244.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
1019,content_f94fe855380e150f,247,6,0.024291,241.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description
766,content_f94fe855380e150f,240,5,0.020833,235.0,HIGH_IMPRESSIONS_LOW_CTR,Review Title and Meta Description


| Rank  | Action                            | Reason Code              | Confidence Note                                                                                | What would make it wrong                                                                                           |
| ----- | --------------------------------- | ------------------------ | ---------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------ |
| 1     | Review Title and Meta Description | HIGH_IMPRESSIONS_LOW_CTR | High confidence because impressions are high and CTR is low.                                   | If low CTR is mainly caused by poor search ranking instead of the page title or description.                       |
| 2     | Review Title and Meta Description | HIGH_IMPRESSIONS_LOW_CTR | High confidence because the page has many missed click opportunities.                          | If impressions come from unrelated search queries.                                                                 |
| 3     | Review Title and Meta Description | HIGH_IMPRESSIONS_LOW_CTR | High confidence based on high visibility and low CTR.                                          | If the page has already been recently optimized.                                                                   |
| 4     | Review Title and Meta Description | HIGH_IMPRESSIONS_LOW_CTR | Good optimization opportunity.                                                                 | If the topic is seasonal and current data is temporary.                                                            |
| 5     | Review Title and Meta Description | HIGH_IMPRESSIONS_LOW_CTR | Likely metadata improvement candidate.                                                         | If users intentionally avoid clicking because the search intent differs.                                           |
| 6     | Review Title and Meta Description | HIGH_IMPRESSIONS_LOW_CTR | High priority due to missed clicks.                                                            | If search results recently changed.                                                                                |
| 7     | Review Title and Meta Description | HIGH_IMPRESSIONS_LOW_CTR | Good candidate for review.                                                                     | If click data contains measurement errors.                                                                         |
| 8     | Review Title and Meta Description | HIGH_IMPRESSIONS_LOW_CTR | Strong visibility but weak CTR.                                                                | If competitors temporarily changed their snippets.                                                                 |
| 9     | Review Title and Meta Description | HIGH_IMPRESSIONS_LOW_CTR | Potential quick win.                                                                           | If low CTR is expected for this query type.                                                                        |
| 10    | Review Title and Meta Description | HIGH_IMPRESSIONS_LOW_CTR | High baseline score.                                                                           | If user intent has shifted.                                                                                        |
| 11–20 | Review Title and Meta Description | HIGH_IMPRESSIONS_LOW_CTR | Moderate confidence because they still have relatively high impressions and below-average CTR. | Would be wrong if the low CTR is explained by ranking position, seasonality, or query intent rather than metadata. |


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks:

Some pages may appear in the ranked list because they have high impressions but naturally low CTR for their search intent. Other pages may be affected by seasonal traffic or temporary ranking changes instead of poor metadata. These pages should be reviewed manually before taking action.

Leakage Check:

No future information was used when calculating the baseline score. The rule only uses current search performance features (gsc_impressions and gsc_clicks to calculate CTR). No target labels, future windows, or product-generated flags were included in the scoring process.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.